## Data preprocessing configuration


# 1 .  Initialize Preprocessing Configuration

In [2]:
%pip install sentencepiece protobuf

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().parents[1]

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "squad_v2"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "squad_v2"

# Create processed-data directory
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Tokenization configuration
MAX_LENGTH = 384
DOC_STRIDE = 128

# Selected model checkpoints
MODEL_CHECKPOINTS = {
    "bert": "bert-base-uncased",
    "roberta": "roberta-base",
    "deberta": "microsoft/deberta-v3-base",
    "distilbert": "distilbert-base-uncased"
}

print("Project root:", PROJECT_ROOT)
print("Raw SQuAD path:", RAW_PATH)
print("Raw folder exists:", RAW_PATH.exists())
print("Processed path:", PROCESSED_PATH)
print("Max sequence length:", MAX_LENGTH)
print("Document stride:", DOC_STRIDE)
print("Random seed:", RANDOM_SEED)
print("Models:", list(MODEL_CHECKPOINTS.keys()))

Project root: /Users/deepika/Desktop/Domain-Adaptive-and-Robustness-Aware-Question-Answering-
Raw SQuAD path: /Users/deepika/Desktop/Domain-Adaptive-and-Robustness-Aware-Question-Answering-/data/raw/squad_v2
Raw folder exists: True
Processed path: /Users/deepika/Desktop/Domain-Adaptive-and-Robustness-Aware-Question-Answering-/data/processed/squad_v2
Max sequence length: 384
Document stride: 128
Random seed: 42
Models: ['bert', 'roberta', 'deberta', 'distilbert']


 # Load SQuAD 2.0 for preprocessing 

In [4]:
# Define actual Parquet file paths
TRAIN_FILE = RAW_PATH / "train" / "0000.parquet"
VALIDATION_FILE = RAW_PATH / "validation" / "0000.parquet"

# Verify files
assert TRAIN_FILE.exists(), f"Training file not found: {TRAIN_FILE}"
assert VALIDATION_FILE.exists(), f"Validation file not found: {VALIDATION_FILE}"

# Load datasets
train_df = pd.read_parquet(TRAIN_FILE)
validation_df = pd.read_parquet(VALIDATION_FILE)

print("Training dataset shape:", train_df.shape)
print("Validation dataset shape:", validation_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nTraining data sample:")
display(train_df.head(2))

print("\nFirst answer annotation:")
print(train_df.iloc[0]["answers"])

Training dataset shape: (130319, 5)
Validation dataset shape: (11873, 5)

Columns:
['id', 'title', 'context', 'question', 'answers']

Training data sample:


,id,title,context,question,answers
0,56be85543aeaaa14008c9063,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce start becoming popular?,"{'text': ['in the late 1990s'], 'answer_start'..."
1,56be85543aeaaa14008c9065,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,What areas did Beyonce compete in when she was...,"{'text': ['singing and dancing'], 'answer_star..."



First answer annotation:
{'text': array(['in the late 1990s'], dtype=object), 'answer_start': array([269], dtype=int32)}


# Standardize answer annotations

In [5]:
def standardize_answers(df):

    df = df.copy()

    # Extract all answer annotations
    df["answer_texts"] = df["answers"].apply(
        lambda x: list(x["text"]) if x is not None else []
    )

    df["answer_starts"] = df["answers"].apply(
        lambda x: list(x["answer_start"]) if x is not None else []
    )

    # Determine answerability
    df["is_impossible"] = df["answer_texts"].apply(
        lambda x: len(x) == 0
    )

    # Extract primary answer and its original character offset
    df["answer_text"] = df["answer_texts"].apply(
        lambda x: x[0] if x else ""
    )

    df["answer_start"] = df["answer_starts"].apply(
        lambda x: x[0] if x else -1
    )

    return df


# Standardize both splits
train_processed = standardize_answers(train_df)
validation_processed = standardize_answers(validation_df)


# Verify results
print("Training shape:", train_processed.shape)
print("Validation shape:", validation_processed.shape)

print("\nNew columns:")
print(train_processed.columns.tolist())

print("\nAnswerable training examples:",
      (~train_processed["is_impossible"]).sum())

print("Unanswerable training examples:",
      train_processed["is_impossible"].sum())

print("\nSample standardized records:")
display(
    train_processed[
        ["question", "answer_text", "answer_start",
         "is_impossible", "answer_texts", "answer_starts"]
    ].head(3)
)

print("\nSample unanswerable record:")
display(
    train_processed[train_processed["is_impossible"]][
        ["question", "answer_text", "answer_start", "is_impossible"]
    ].head(1)
)

Training shape: (130319, 10)
Validation shape: (11873, 10)

New columns:
['id', 'title', 'context', 'question', 'answers', 'answer_texts', 'answer_starts', 'is_impossible', 'answer_text', 'answer_start']

Answerable training examples: 86821
Unanswerable training examples: 43498

Sample standardized records:


,question,answer_text,answer_start,is_impossible,answer_texts,answer_starts
0,When did Beyonce start becoming popular?,in the late 1990s,269,False,[in the late 1990s],[269]
1,What areas did Beyonce compete in when she was...,singing and dancing,207,False,[singing and dancing],[207]
2,When did Beyonce leave Destiny's Child and bec...,2003,526,False,[2003],[526]



Sample unanswerable record:


,question,answer_text,answer_start,is_impossible
2075,What category of game is Legend of Zelda: Aust...,,-1,True


# Validate answer character offsets

In [6]:
def validate_answer_offsets(df, split_name):

    total_annotations = 0
    valid_offsets = 0
    invalid_offsets = 0
    invalid_examples = []

    for _, row in df.iterrows():

        if row["is_impossible"]:
            continue

        context = row["context"]

        for answer, start in zip(
            row["answer_texts"],
            row["answer_starts"]
        ):
            total_annotations += 1

            extracted_text = context[start:start + len(answer)]

            if extracted_text == answer:
                valid_offsets += 1
            else:
                invalid_offsets += 1

                invalid_examples.append({
                    "id": row["id"],
                    "answer": answer,
                    "answer_start": start,
                    "extracted_text": extracted_text
                })

    print(f"\n{split_name.upper()} OFFSET VALIDATION")
    print("Total answer annotations:", total_annotations)
    print("Valid offsets:", valid_offsets)
    print("Invalid offsets:", invalid_offsets)

    if total_annotations:
        print(
            "Validation accuracy:",
            round(valid_offsets / total_annotations * 100, 2),
            "%"
        )

    return pd.DataFrame(invalid_examples)


# Validate both splits
train_invalid_offsets = validate_answer_offsets(
    train_processed, "Training"
)

validation_invalid_offsets = validate_answer_offsets(
    validation_processed, "Validation"
)

print("\nInvalid training examples:")
display(train_invalid_offsets.head(10))

print("\nInvalid validation examples:")
display(validation_invalid_offsets.head(10))


TRAINING OFFSET VALIDATION
Total answer annotations: 86821
Valid offsets: 86821
Invalid offsets: 0
Validation accuracy: 100.0 %

VALIDATION OFFSET VALIDATION
Total answer annotations: 20302
Valid offsets: 20302
Invalid offsets: 0
Validation accuracy: 100.0 %

Invalid training examples:


""



Invalid validation examples:


""


# Load pretrained tokenizers

In [7]:
from transformers import AutoTokenizer

MODEL_CHECKPOINTS = {
    "bert": "bert-base-uncased",
    "roberta": "roberta-base",
    "deberta": "microsoft/deberta-v3-base",
    "distilbert": "distilbert-base-uncased"
}

tokenizers = {}

for model_name, checkpoint in MODEL_CHECKPOINTS.items():
    tokenizers[model_name] = AutoTokenizer.from_pretrained(
        checkpoint,
        use_fast=True
    )

    print(
        model_name,
        "| Fast tokenizer:",
        tokenizers[model_name].is_fast
    )

bert | Fast tokenizer: True
roberta | Fast tokenizer: True
deberta | Fast tokenizer: True
distilbert | Fast tokenizer: True


# Tokenize questions and contexts

In [14]:

# Tokenize the complete SQuAD datasets for each model

from tqdm.auto import tqdm

def tokenize_dataset(df, tokenizer, batch_size=500):
    all_features = []

    for start in tqdm(range(0, len(df), batch_size)):
        batch = df.iloc[start:start + batch_size]

        encoded = tokenizer(
            batch["question"].tolist(),
            batch["context"].tolist(),
            max_length=MAX_LENGTH,
            stride=DOC_STRIDE,
            truncation="only_second",
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding=False
        )

        all_features.append(encoded)

    return all_features


# Start with BERT first
bert_train_tokens = tokenize_dataset(
    train_processed, tokenizers["bert"]
)

bert_validation_tokens = tokenize_dataset(
    validation_processed, tokenizers["bert"]
)

print("BERT tokenization completed")
print("Training batches:", len(bert_train_tokens))
print("Validation batches:", len(bert_validation_tokens))
print("First batch features:", len(bert_train_tokens[0]["input_ids"]))
print("First feature token length:", len(bert_train_tokens[0]["input_ids"][0]))
print("Offset mappings available:", "offset_mapping" in bert_train_tokens[0])


  0%|          | 0/261 [00:00<?, ?it/s]

  0%|          | 0/24 [00:00<?, ?it/s]

BERT tokenization completed
Training batches: 261
Validation batches: 24
First batch features: 527
First feature token length: 174
Offset mappings available: True


# Character-to-token answer alignment

In [12]:
# Extract answer annotations from nested answers column

def extract_answers(answer_dict):
    return {
        "answer_texts": answer_dict["text"],
        "answer_starts": answer_dict["answer_start"]
    }

for df in [train_df, validation_df]:
    extracted = df["answers"].apply(extract_answers).apply(pd.Series)
    df["answer_texts"] = extracted["answer_texts"]
    df["answer_starts"] = extracted["answer_starts"]

# Verify extraction
print("Training columns:", train_df.columns.tolist())
print("Validation columns:", validation_df.columns.tolist())

display(train_df[["question", "answer_texts", "answer_starts"]].head())

Training columns: ['id', 'title', 'context', 'question', 'answers', 'answer_texts', 'answer_starts']
Validation columns: ['id', 'title', 'context', 'question', 'answers', 'answer_texts', 'answer_starts']


,question,answer_texts,answer_starts
0,When did Beyonce start becoming popular?,[in the late 1990s],[269]
1,What areas did Beyonce compete in when she was...,[singing and dancing],[207]
2,When did Beyonce leave Destiny's Child and bec...,[2003],[526]
3,In what city and state did Beyonce grow up?,"[Houston, Texas]",[166]
4,In which decade did Beyonce become famous?,[late 1990s],[276]


Create unanswerable question flags

In [13]:
# Create unanswerable flags

for df in [train_df, validation_df]:
    df["is_impossible"] = df["answer_texts"].apply(
        lambda answers: len(answers) == 0
    )

# Verify distribution
print("TRAINING ANSWERABILITY:")
print(train_df["is_impossible"].value_counts())

print("\nVALIDATION ANSWERABILITY:")
print(validation_df["is_impossible"].value_counts())

# Display examples
print("\nUNANSWERABLE EXAMPLE:")
display(
    train_df[train_df["is_impossible"]][
        ["question", "answer_texts", "is_impossible"]
    ].head(3)
)

TRAINING ANSWERABILITY:
is_impossible
False    86821
True     43498
Name: count, dtype: int64

VALIDATION ANSWERABILITY:
is_impossible
True     5945
False    5928
Name: count, dtype: int64

UNANSWERABLE EXAMPLE:


,question,answer_texts,is_impossible
2075,What category of game is Legend of Zelda: Aust...,[],True
2076,What consoles can be used to play Australia Tw...,[],True
2077,When was Australia Twilight launched in North ...,[],True


# Validate answer start/end positions using gold-span round-trip tests.

# Investigate and document duplicate records and annotation conflicts.

In [20]:
def answer_signature(row):
    return (
        tuple(row["answer_texts"]),
        tuple(int(x) for x in row["answer_starts"])
    )


def investigate_duplicates(df, name):
    data = df.copy()

    data["answer_signature"] = data.apply(answer_signature, axis=1)

    print(f"\n{'='*45}")
    print(name)
    print(f"{'='*45}")

    # 1. Duplicate IDs
    print("Duplicate IDs:", data["id"].duplicated().sum())

    # 2. Duplicate question-context pairs
    pair_duplicates = data.duplicated(
        subset=["question", "context"]
    ).sum()
    print("Duplicate question-context pairs:", pair_duplicates)

    # 3. Exact duplicate records
    exact_duplicates = data.duplicated(
        subset=["id", "title", "context", "question", "answer_signature"]
    ).sum()
    print("Exact duplicate records:", exact_duplicates)

    # 4. Conflicting annotations for identical question-context pairs
    conflicts = (
        data.groupby(["question", "context"])["answer_signature"]
        .nunique()
    )
    conflicts = conflicts[conflicts > 1]

    print("Annotation conflict groups:", len(conflicts))

    if len(conflicts) > 0:
        print("\nConflict examples:")
        display(
            data[
                data.set_index(["question", "context"]).index.isin(conflicts.index)
            ][["question", "answer_texts", "answer_starts"]].head(5)
        )


investigate_duplicates(train_df, "TRAINING")
investigate_duplicates(validation_df, "VALIDATION")


TRAINING
Duplicate IDs: 0
Duplicate question-context pairs: 64
Exact duplicate records: 0
Annotation conflict groups: 38

Conflict examples:


,question,answer_texts,answer_starts
1226,Who orchestrated Chopiniana?,[Alexander Glazunov],[106]
1230,Who orchestrated Chopiniana?,[Alexander Glazunov.],[106]
1255,Where was Chopin's last public performance?,[Guildhall],[73]
1258,Where was Chopin's last public performance?,[London's Guildhall],[64]
1285,Where was Chopin's funeral held?,[Church of the Madeleine],[25]



VALIDATION
Duplicate IDs: 0
Duplicate question-context pairs: 7
Exact duplicate records: 0
Annotation conflict groups: 6

Conflict examples:


,question,answer_texts,answer_starts
2022,Who designed Salamanca?,"[Matthew Murray, Matthew Murray, Matthew Murray]","[192, 192, 192]"
2027,Who designed Salamanca?,[],[]
2266,Where does heat rejection occur in the Rankine...,"[in the condenser, in the condenser, in the co...","[240, 240, 240]"
2271,Where does heat rejection occur in the Rankine...,[],[]
2946,What are the main sources of primary law?,"[Treaties establishing the European Union, the...","[317, 313, 227, 313]"


In [21]:
def show_conflicts(df, name):
    signatures = df.apply(answer_signature, axis=1)

    temp = df.copy()
    temp["answer_signature"] = signatures

    conflict_groups = (
        temp.groupby(["question", "context"])["answer_signature"]
        .nunique()
    )
    conflict_groups = conflict_groups[conflict_groups > 1]

    print(f"\n{name}: {len(conflict_groups)} conflict groups")

    for question, context in conflict_groups.index:
        rows = temp[
            (temp["question"] == question) &
            (temp["context"] == context)
        ]

        print("\nQuestion:", question)
        for _, row in rows.iterrows():
            print("ID:", row["id"])
            print("Answers:", row["answer_texts"])
            print("Starts:", row["answer_starts"])


show_conflicts(train_df, "TRAINING")
show_conflicts(validation_df, "VALIDATION")


TRAINING: 38 conflict groups

Question: How many men's intercollegiate athletic teams are fielded by Northwestern?
ID: 57280ae14b864d19001642d2
Answers: ['8']
Starts: [55]
ID: 5ad2d01cd7d075001a42a32f
Answers: []
Starts: []

Question: How many women's intercollegiate athletic teams are fielded by Northwestern?
ID: 57280ae14b864d19001642d3
Answers: ['11']
Starts: [67]
ID: 5ad2d01cd7d075001a42a32e
Answers: []
Starts: []

Question: What area of China has the least amount of sunlight?
ID: 5727b5b5ff5b5019007d9317
Answers: ['the Sichuan Basin']
Starts: [223]
ID: 5a514a03ce860b001aa3fcb6
Answers: []
Starts: []

Question: What borders South Raleigh to the west?
ID: 5726b298f1498d1400e8e7d2
Answers: ['Cary,']
Starts: [345]
ID: 5acd59a007355d001abf3e3d
Answers: []
Starts: []

Question: What did Avicenna consider to be the only way to distinguish real philosophy from illusion?
ID: 5726e3e2f1498d1400e8eef2
Answers: ['philosophy']
Starts: [100]
ID: 5acea28132bba1001ae4ae38
Answers: []
Starts: []


In [22]:
def classify_conflicts(df, name):
    temp = df.copy()

    temp["has_answer"] = temp["answer_texts"].apply(
        lambda x: len(x) > 0
    )

    groups = temp.groupby(["question", "context"])

    answerability_conflicts = 0
    answer_variation_groups = 0
    repeated_annotation_groups = 0

    for _, group in groups:
        if len(group) < 2:
            continue

        if group["has_answer"].nunique() > 1:
            answerability_conflicts += 1

        nonempty = [
            tuple(sorted(set(str(a).strip().lower() for a in answers)))
            for answers in group["answer_texts"]
            if len(answers) > 0
        ]

        if len(set(nonempty)) > 1:
            answer_variation_groups += 1

        if any(len(answers) > len(set(answers))
               for answers in group["answer_texts"]):
            repeated_annotation_groups += 1

    print(f"\n{name}")
    print("Answerability conflicts:", answerability_conflicts)
    print("Different answer groups:", answer_variation_groups)
    print("Groups with repeated annotations:", repeated_annotation_groups)


classify_conflicts(train_df, "TRAINING")
classify_conflicts(validation_df, "VALIDATION")


TRAINING
Answerability conflicts: 25
Different answer groups: 13
Groups with repeated annotations: 0

VALIDATION
Answerability conflicts: 4
Different answer groups: 2
Groups with repeated annotations: 6


# Check dataset and document leakage across splits

In [23]:
# Check overlap between training and validation

train_ids = set(train_df["id"])
val_ids = set(validation_df["id"])

train_questions = set(train_df["question"])
val_questions = set(validation_df["question"])

train_contexts = set(train_df["context"])
val_contexts = set(validation_df["context"])

print("ID overlap:", len(train_ids & val_ids))
print("Question overlap:", len(train_questions & val_questions))
print("Context overlap:", len(train_contexts & val_contexts))

ID overlap: 0
Question overlap: 7
Context overlap: 0


In [24]:
train_questions = set(train_df["question"].dropna())
val_questions = set(validation_df["question"].dropna())

overlap_questions = train_questions & val_questions

for question in sorted(overlap_questions):
    print("\nQuestion:", question)

    print("TRAIN:")
    display(train_df[train_df["question"] == question][
        ["id", "context", "answer_texts"]
    ])

    print("VALIDATION:")
    display(validation_df[validation_df["question"] == question][
        ["id", "context", "answer_texts"]
    ])


Question: How long is the Rhine?
TRAIN:


,id,context,answer_texts
83138,5acd751c07355d001abf4344,"At present the A35 autoroute, which parallels ...",[]


VALIDATION:


,id,context,answer_texts
9008,572f5533a23a5019007fc55f,"The Rhine (Romansh: Rein, German: Rhein, Frenc...","[1,230 km (760 mi), 1,230 km (760 mi), 1,230 k..."



Question: In what year did Louis XIV die?
TRAIN:


,id,context,answer_texts
45463,57103d79b654c5140001f8bb,French historians traditionally place the Enli...,[1715]


... (18 outputs left out)


Question: When was the first Sino-Japanese War?
TRAIN:


,id,context,answer_texts
119607,57311e26497a881900248b71,The reign of the Qianlong Emperor (1735–1796) ...,[1895]


VALIDATION:


,id,context,answer_texts
10525,5730b1022461fd1900a9cfa3,"During the First Sino-Japanese War in 1894, Ja...","[1894, 1894, 1894, 1894, 1894]"


In [25]:
for question in sorted(overlap_questions):
    train_rows = train_df[train_df["question"] == question]
    val_rows = validation_df[validation_df["question"] == question]

    same_context = set(train_rows["context"]) & set(val_rows["context"])

    print(f"Question: {question}")
    print("Same context across splits:", bool(same_context))
    print("-" * 60)

Question: How long is the Rhine?
Same context across splits: False
------------------------------------------------------------
Question: In what year did Louis XIV die?
Same context across splits: False
------------------------------------------------------------
Question: What does CSA stand for?
Same context across splits: False
------------------------------------------------------------
Question: What is the population of the Netherlands?
Same context across splits: False
------------------------------------------------------------
Question: What was the average family size?
Same context across splits: False
------------------------------------------------------------
Question: What was the capital of the Ottoman empire?
Same context across splits: False
------------------------------------------------------------
Question: When was the first Sino-Japanese War?
Same context across splits: False
------------------------------------------------------------


# Create reproducible train/validation splits and fixed few-shot subsets.

In [26]:
print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

print("\nTraining columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(validation_df.columns.tolist())

print("\nTraining answerability:")
print(train_df["answer_texts"].apply(len).gt(0).value_counts())

print("\nValidation answerability:")
print(validation_df["answer_texts"].apply(len).gt(0).value_counts())

Training shape: (130319, 8)
Validation shape: (11873, 8)

Training columns:
['id', 'title', 'context', 'question', 'answers', 'answer_texts', 'answer_starts', 'is_impossible']

Validation columns:
['id', 'title', 'context', 'question', 'answers', 'answer_texts', 'answer_starts', 'is_impossible']

Training answerability:
answer_texts
True     86821
False    43498
Name: count, dtype: int64

Validation answerability:
answer_texts
False    5945
True     5928
Name: count, dtype: int64


In [27]:
# Create reproducible, stratified training subsets

SEED = 42
fractions = [0.01, 0.05, 0.10]

# Shuffle each answerability group once
groups = {
    label: group.sample(frac=1, random_state=SEED)
    for label, group in train_df.groupby("is_impossible")
}

few_shot_subsets = {}

for fraction in fractions:
    sampled_groups = []

    for label, group in groups.items():
        n_samples = round(len(group) * fraction)
        sampled_groups.append(group.iloc[:n_samples])

    subset = (
        pd.concat(sampled_groups)
        .sample(frac=1, random_state=SEED)
        .reset_index(drop=True)
    )

    few_shot_subsets[f"{int(fraction * 100)}%"] = subset

    print(f"\n{int(fraction * 100)}% TRAINING SUBSET")
    print("Total examples:", len(subset))
    print("Answerable:", (~subset["is_impossible"]).sum())
    print("Unanswerable:", subset["is_impossible"].sum())

print("\nOriginal validation size:", len(validation_df))


1% TRAINING SUBSET
Total examples: 1303
Answerable: 868
Unanswerable: 435

5% TRAINING SUBSET
Total examples: 6516
Answerable: 4341
Unanswerable: 2175

10% TRAINING SUBSET
Total examples: 13032
Answerable: 8682
Unanswerable: 4350

Original validation size: 11873


In [28]:
# Verify nested subsets using unique IDs

ids_1 = set(few_shot_subsets["1%"]["id"])
ids_5 = set(few_shot_subsets["5%"]["id"])
ids_10 = set(few_shot_subsets["10%"]["id"])

print("1% subset contained in 5%:", ids_1.issubset(ids_5))
print("5% subset contained in 10%:", ids_5.issubset(ids_10))

# Verify reproducibility
repeat_groups = {
    label: group.sample(frac=1, random_state=SEED)
    for label, group in train_df.groupby("is_impossible")
}

repeat_1 = pd.concat([
    group.iloc[:round(len(group) * 0.01)]
    for group in repeat_groups.values()
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(
    "Reproducibility check:",
    repeat_1["id"].tolist() == few_shot_subsets["1%"]["id"].tolist()
)

1% subset contained in 5%: True
5% subset contained in 10%: True
Reproducibility check: True


# # Save processed datasets and preprocessing metadata in data/processed/

In [34]:

from pathlib import Path
import json
from datetime import datetime
import pandas as pd

# Reproducible random seed
RANDOM_STATE = 42

# Create output directory
OUTPUT_DIR = Path("data/processed/squad2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create 5% subset from existing 10% subset
remaining = subset[~subset["id"].isin(repeat_1["id"])]

train_5 = pd.concat([
    repeat_1,
    remaining.sample(
        n=6516 - len(repeat_1),
        random_state=RANDOM_STATE
    )
]).sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

# Collect datasets
datasets = {
    "train_1_percent": repeat_1,
    "train_5_percent": train_5,
    "train_10_percent": subset,
    "validation": validation_df
}

# Save datasets
for name, df in datasets.items():
    path = OUTPUT_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved {name}: {df.shape}")

# Save metadata
metadata = {
    "dataset": "SQuAD 2.0",
    "random_seed": RANDOM_STATE,
    "subsets": {
        name: len(df) for name, df in datasets.items()
    },
    "created_at": datetime.now().isoformat()
}

with open(OUTPUT_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("\nStep 13 saving completed.")


Saved train_1_percent: (1303, 8)
Saved train_5_percent: (6516, 8)
Saved train_10_percent: (13032, 8)
Saved validation: (11873, 8)

Step 13 saving completed.


# Run preprocessing for BERT, RoBERTa, DeBERTa and DistilBERT

In [35]:
%pip install transformers datasets sentencepiece pyarrow

Note: you may need to restart the kernel to use updated packages.


# Final Preprocessing Validation and Handoff

In [41]:

from pathlib import Path
from transformers import AutoTokenizer
from datasets import load_dataset
import pandas as pd

# Paths
INPUT_DIR = Path("data/processed/squad2")
OUTPUT_DIR = Path("data/processed/squad2/tokenized")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Models
MODELS = {
    "bert": "bert-base-uncased",
    "roberta": "roberta-base",
    "deberta": "microsoft/deberta-v3-base",
    "distilbert": "distilbert-base-uncased"
}

MAX_LENGTH = 384
STRIDE = 128

# Input datasets
FILES = {
    "train_1_percent": INPUT_DIR / "train_1_percent.parquet",
    "train_5_percent": INPUT_DIR / "train_5_percent.parquet",
    "train_10_percent": INPUT_DIR / "train_10_percent.parquet",
    "validation": INPUT_DIR / "validation.parquet"
}

# Load datasets
datasets = {
    name: load_dataset(
        "parquet",
        data_files=str(path),
        split="train"
    )
    for name, path in FILES.items()
}

# Tokenization and answer alignment
def preprocess_function(examples, tokenizer):

    questions = [q.strip() for q in examples["question"]]

    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        stride=STRIDE,
        truncation="only_second",
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    raw_offsets = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []
    example_ids = []
    saved_offsets = []
    is_impossible = []

    for i, offsets in enumerate(raw_offsets):

        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        example_ids.append(str(examples["id"][sample_index]))

        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized.sequence_ids(i)

        # Keep offsets only for context tokens
        context_positions = [
            j for j, sid in enumerate(sequence_ids)
            if sid == 1
        ]

        clean_offsets = [
            list(offsets[j]) if sequence_ids[j] == 1 else [-1, -1]
            for j in range(len(offsets))
        ]

        saved_offsets.append(clean_offsets)

        # Unanswerable question
        if not answers["answer_start"] or not answers["text"]:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            is_impossible.append(True)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        # No context tokens in this window
        if not context_positions:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            is_impossible.append(False)
            continue

        # Locate answer boundary tokens
        start_token = next(
            (
                j for j in context_positions
                if offsets[j][0] <= start_char < offsets[j][1]
            ),
            None
        )

        end_token = next(
            (
                j for j in reversed(context_positions)
                if offsets[j][0] < end_char <= offsets[j][1]
            ),
            None
        )

        # Answer not fully present in this window
        if (
            start_token is None
            or end_token is None
            or start_token > end_token
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            is_impossible.append(False)
        else:
            start_positions.append(start_token)
            end_positions.append(end_token)
            is_impossible.append(False)

    tokenized["example_id"] = example_ids
    tokenized["offset_mapping"] = saved_offsets
    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    tokenized["is_impossible"] = is_impossible

    return tokenized


# Process all datasets with all models
for model_name, checkpoint in MODELS.items():

    print(f"\nProcessing: {model_name.upper()}")

    tokenizer = AutoTokenizer.from_pretrained(
        checkpoint,
        use_fast=True
    )

    if tokenizer.cls_token_id is None:
        raise ValueError(f"{model_name} has no CLS token ID")

    for dataset_name, dataset in datasets.items():

        processed = dataset.map(
            lambda examples: preprocess_function(
                examples, tokenizer
            ),
            batched=True,
            remove_columns=dataset.column_names
        )

        # Final structural validation
        invalid = 0
        too_long = 0

        for row in processed:
            ids = row["input_ids"]
            start = row["start_positions"]
            end = row["end_positions"]

            if len(ids) > MAX_LENGTH:
                too_long += 1

            if not (0 <= start <= end < len(ids)):
                invalid += 1

        if invalid or too_long:
            raise ValueError(
                f"Validation failed: {model_name}/{dataset_name}"
            )

        # Save updated file
        output_path = (
            OUTPUT_DIR / model_name / f"{dataset_name}.parquet"
        )
        output_path.parent.mkdir(parents=True, exist_ok=True)

        processed.to_parquet(str(output_path))

        print(
            f"{dataset_name}: {processed.num_rows} features | "
            f"Invalid positions: {invalid} | "
            f"Overlength sequences: {too_long}"
        )

print("\nStep 15 completed: all tokenized datasets regenerated.")



Processing: BERT


Map:   0%|          | 0/1303 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

train_1_percent: 1321 features | Invalid positions: 0 | Overlength sequences: 0


Map:   0%|          | 0/6516 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

... (39 outputs left out)

train_10_percent: 13164 features | Invalid positions: 0 | Overlength sequences: 0


Map:   0%|          | 0/11873 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

validation: 12134 features | Invalid positions: 0 | Overlength sequences: 0

Step 15 completed: all tokenized datasets regenerated.


In [43]:

from pathlib import Path
import pandas as pd
from transformers import AutoTokenizer

RAW_PROCESSED = Path("data/processed/squad2")
TOKENIZED = RAW_PROCESSED / "tokenized"

MODELS = {
    "bert": "bert-base-uncased",
    "roberta": "roberta-base",
    "deberta": "microsoft/deberta-v3-base",
    "distilbert": "distilbert-base-uncased"
}

DATASETS = {
    "train_1_percent": RAW_PROCESSED / "train_1_percent.parquet",
    "train_5_percent": RAW_PROCESSED / "train_5_percent.parquet",
    "train_10_percent": RAW_PROCESSED / "train_10_percent.parquet",
    "validation": RAW_PROCESSED / "validation.parquet"
}


def get_gold_answers(row):
    # Original nested SQuAD answer structure
    if "answers" in row and isinstance(row["answers"], dict):
        return (
            row["answers"].get("text", []),
            row["answers"].get("answer_start", [])
        )

    # Standardized answer columns
    texts = row.get("answer_texts", [])
    starts = row.get("answer_starts", [])

    if not isinstance(texts, list):
        texts = [texts] if pd.notna(texts) else []

    if not isinstance(starts, list):
        starts = [starts] if pd.notna(starts) else []

    return texts, starts


total_passed = 0
total_failed = 0

for model_name, checkpoint in MODELS.items():

    print(f"\n{'=' * 60}")
    print(f"MODEL: {model_name.upper()}")
    print("=" * 60)

    tokenizer = AutoTokenizer.from_pretrained(
        checkpoint,
        use_fast=True
    )

    cls_id = tokenizer.cls_token_id

    for dataset_name, original_path in DATASETS.items():

        original_df = pd.read_parquet(original_path)
        tokenized_path = (
            TOKENIZED / model_name / f"{dataset_name}.parquet"
        )
        tokenized_df = pd.read_parquet(tokenized_path)

        # Original examples indexed by their IDs
        original_df["id"] = original_df["id"].astype(str)
        original_lookup = original_df.set_index("id").to_dict("index")

        passed = 0
        failed = 0
        checked = 0

        for _, feature in tokenized_df.iterrows():

            example_id = str(feature["example_id"])

            if example_id not in original_lookup:
                failed += 1
                continue

            original = original_lookup[example_id]

            start = int(feature["start_positions"])
            end = int(feature["end_positions"])
            input_ids = feature["input_ids"]

            # CLS-labelled feature: no answer span in this window
            if (
                input_ids[start] == cls_id
                and input_ids[end] == cls_id
            ):
                continue

            offsets = feature["offset_mapping"]

            start_char = offsets[start][0]
            end_char = offsets[end][1]

            if start_char < 0 or end_char <= start_char:
                failed += 1
                continue

            context = original["context"]
            reconstructed = context[start_char:end_char]

            answer_texts, answer_starts = get_gold_answers(original)

            # Verify against the original gold annotation
            matches = any(
                reconstructed == text
                and start_char == answer_start
                and end_char == answer_start + len(text)
                for text, answer_start in zip(
                    answer_texts, answer_starts
                )
            )

            checked += 1

            if matches:
                passed += 1
            else:
                failed += 1

        total_passed += passed
        total_failed += failed

        print(
            f"{dataset_name}: "
            f"Checked={checked} | "
            f"Passed={passed} | "
            f"Failed={failed}"
        )

print("\n" + "=" * 60)
print("FINAL GOLD ANSWER-SPAN VERIFICATION")
print("=" * 60)
print("Total passed:", total_passed)
print("Total failed:", total_failed)

if total_failed == 0:
    print("RESULT: ALL CHECKED ANSWER SPANS PASSED")
else:
    print("RESULT: INVESTIGATION REQUIRED")




MODEL: BERT
train_1_percent: Checked=871 | Passed=869 | Failed=2
train_5_percent: Checked=4390 | Passed=4381 | Failed=9
train_10_percent: Checked=8700 | Passed=8682 | Failed=18
validation: Checked=5953 | Passed=5951 | Failed=2

MODEL: ROBERTA
train_1_percent: Checked=870 | Passed=854 | Failed=16
train_5_percent: Checked=4389 | Passed=4319 | Failed=70
train_10_percent: Checked=8699 | Passed=8573 | Failed=126
validation: Checked=5952 | Passed=5904 | Failed=48

MODEL: DEBERTA
train_1_percent: Checked=870 | Passed=69 | Failed=801
train_5_percent: Checked=4385 | Passed=345 | Failed=4040
train_10_percent: Checked=8692 | Passed=654 | Failed=8038
validation: Checked=5951 | Passed=364 | Failed=5587

MODEL: DISTILBERT
train_1_percent: Checked=871 | Passed=869 | Failed=2
train_5_percent: Checked=4390 | Passed=4381 | Failed=9
train_10_percent: Checked=8700 | Passed=8682 | Failed=18
validation: Checked=5953 | Passed=5951 | Failed=2

FINAL GOLD ANSWER-SPAN VERIFICATION
Total passed: 60848
Total fai